<a href="https://colab.research.google.com/github/pkrishnateja1/Credit_Risk_Analysis_EDA/blob/main/Fake_News_Detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Fake News Detection using TensorFlow (LSTM)

### Objective:
- Detect fake and real news articles using Deep Learning (LSTM)
- Dataset: Fake News Dataset (news.csv)


####Step 1: Install Required Libraries (Uncomment if needed)


In [ ]:
#!pip install tensorflow pandas numpy scikit-learn --quiet

####Step 2: Import Required Libraries

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

print("Libraries imported successfully.")


Libraries imported successfully.


####Step 3: Load Dataset

- Ensure the dataset file news.csv is in the same folder as this notebook.

In [ ]:
DATA_PATH = "/news.csv"  # Update path if needed

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


Dataset loaded successfully.
Shape: (6335, 4)


,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


####Step 4: Identify Text and Label Columns

- Automatically detect the columns that contain the news text and labels.

In [ ]:
text_col = None
label_col = None

for c in ['text', 'article', 'content', 'body', 'news', 'headline', 'title']:
    if c in df.columns:
        text_col = c
        break

if text_col is None and 'title' in df.columns and 'text' in df.columns:
    df['combined'] = df['title'].astype(str) + " " + df['text'].astype(str)
    text_col = 'combined'
elif text_col is None and 'title' in df.columns:
    text_col = 'title'

for c in ['label', 'class', 'target', 'truth', 'type']:
    if c in df.columns:
        label_col = c
        break

if label_col is None:
    possible = [c for c in df.columns if c != text_col]
    label_col = possible[-1]

print("Text column:", text_col)
print("Label column:", label_col)


Text column: text
Label column: label


####Step 5: Encode Labels (FAKE = 0, REAL = 1)

- Convert categorical labels into binary numeric values.

In [ ]:
df[label_col] = df[label_col].astype(str).str.lower().map({'fake': 0, 'real': 1})
if df[label_col].isnull().any():
    df[label_col] = pd.to_numeric(df[label_col], errors='coerce')

df.dropna(subset=[label_col], inplace=True)
df[label_col] = df[label_col].astype(int)

print("Label encoding completed.")
print(df[label_col].value_counts())


Label encoding completed.
label
1    3171
0    3164
Name: count, dtype: int64


####Step 6: Clean and Preprocess Text

Text preprocessing includes:

- Lowercasing

- Removing URLs, punctuation, and stopwords

- Token cleaning

In [ ]:
stopwords = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    tokens = [word for word in text.split() if word not in stopwords and len(word) > 1]
    return " ".join(tokens)

df['clean_text'] = df[text_col].apply(clean_text)
print("Text cleaning completed.")
df[['clean_text']].head()


Text cleaning completed.


,clean_text
0,daniel greenfield shillman journalism fellow f...
1,google pinterest digg linkedin reddit stumbleu...
2,secretary state john kerry said monday stop pa...
3,kaydee king kaydeeking november lesson tonight...
4,primary day new york runners hillary clinton d...


####Step 7: Split Dataset into Training and Testing Sets

In [ ]:
X = df['clean_text'].values
y = df[label_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


Training samples: 5068
Testing samples : 1267


####Step 8: Tokenization and Padding

- Convert text into sequences of numbers and pad them to uniform length for model input.

In [ ]:
vocab_size = 20000
max_len = 300
embedding_dim = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print("Tokenization and padding completed.")
print("Sequence shape:", X_train_pad.shape)


Tokenization and padding completed.
Sequence shape: (5068, 300)


####Step 9: Build the LSTM Model

Architecture:

- Embedding Layer

- Bidirectional LSTM Layer

- Dense Layers with ReLU and Sigmoid activation

- Dropout for regularization

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Bidirectional(LSTM(128, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

####Step 10: Train the Model

- Use Early Stopping and Model Checkpoint callbacks to prevent overfitting and save the best model.

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ModelCheckpoint("/best_fake_news_model.h5", save_best_only=True, monitor='val_loss')
]

history = model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=6,
    batch_size=64,
    callbacks=callbacks,
    verbose=2
)


Epoch 1/6


72/72 - 87s - 1s/step - accuracy: 0.9643 - loss: 0.1108 - val_accuracy: 0.9093 - val_loss: 0.2464
Epoch 2/6
72/72 - 87s - 1s/step - accuracy: 0.9914 - loss: 0.0252 - val_accuracy: 0.9132 - val_loss: 0.3495
Epoch 3/6
72/72 - 139s - 2s/step - accuracy: 0.9985 - loss: 0.0052 - val_accuracy: 0.9034 - val_loss: 0.5174


####Step 11: Evaluate the Model

- Evaluate model performance using accuracy, precision, recall, and F1-score.

In [ ]:
y_pred_probs = model.predict(X_test_pad).ravel()
y_pred = (y_pred_probs >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Evaluation Results:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


40/40 ━━━━━━━━━━━━━━━━━━━━ 9s 221ms/step
Model Evaluation Results:
Accuracy : 0.9266
Precision: 0.9246
Recall   : 0.9290
F1-score : 0.9268

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.92      0.93       633
           1       0.92      0.93      0.93       634

    accuracy                           0.93      1267
   macro avg       0.93      0.93      0.93      1267
weighted avg       0.93      0.93      0.93      1267



####Step 12: Save the Model and Tokenizer

- Save the trained model and tokenizer for future use.

In [ ]:
model.save("/fake_news_lstm_model.h5")
with open("/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully.")


Model and tokenizer saved successfully.


####Step 13: Predict on New Articles

- Test the trained model with new unseen news headlines or articles.

In [ ]:
def predict_news(texts, threshold=0.5):
    cleaned = [clean_text(t) for t in texts]
    seq = tokenizer.texts_to_sequences(cleaned)
    pad_seq = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    probs = model.predict(pad_seq).ravel()
    preds = (probs >= threshold).astype(int)
    labels = {0: "FAKE", 1: "REAL"}
    results = [{"Text": t, "Predicted Label": labels[int(p >= threshold)], "Probability (Real)": float(p)} for t, p in zip(texts, probs)]
    return pd.DataFrame(results)

test_samples = [
    "The government announced a new economic policy today.",
    "Aliens have been found on Mars, NASA confirms!",
    "Scientists discovered a new method to cure flu."
]

pred_results = predict_news(test_samples)
pred_results


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step


,Text,Predicted Label,Probability (Real)
0,The government announced a new economic policy...,FAKE,0.032855
1,"Aliens have been found on Mars, NASA confirms!",FAKE,0.013130
2,Scientists discovered a new method to cure flu.,FAKE,0.065721


####Step 14: Conclusion

This LSTM-based Fake News Detection Model classifies news articles as FAKE or REAL with high accuracy.

**Summary of the Approach:**

- Preprocessing: Tokenization, stopword removal, cleaning text

- Embeddings: Represented words as dense vectors

- Model: Bidirectional LSTM with dropout

- Evaluation: Accuracy, Precision, Recall, F1-score

- Future Improvements: Incorporate transformer-based models like BERT or Attention mechanisms for higher performance.